In [11]:
from dotenv import load_dotenv
load_dotenv()

import os
from openai import OpenAI
from pinecone import Pinecone, ServerlessSpec


In [5]:
pc  = Pinecone(api_key=os.getenv("PINECONE_API_KEY"))
pc.create_index(
    name="rag", dimension=1536, metric='cosine', spec= ServerlessSpec(cloud = 'aws', region = 'us-east-1')
)

In [8]:
import json
data = json.load(open("reviews.json"))
data['reviews']

[{'professor': 'Dr. Emily Richards',
  'subject': 'Physics 101',
  'stars': 4,
  'review': 'Dr. Richards is knowledgeable and her lectures are clear. However, the exams are tough.'},
 {'professor': 'Dr. John Matthews',
  'subject': 'Chemistry 202',
  'stars': 5,
  'review': 'Amazing professor! He makes complex topics easy to understand.'},
 {'professor': 'Dr. Angela Martin',
  'subject': 'Mathematics 305',
  'stars': 3,
  'review': "Her lectures are detailed, but she doesn't explain concepts in a way that's easy to grasp."},
 {'professor': 'Dr. Michael Johnson',
  'subject': 'Biology 150',
  'stars': 5,
  'review': "Dr. Johnson's passion for biology is contagious. Highly recommended!"},
 {'professor': 'Dr. Susan Lee',
  'subject': 'English Literature 210',
  'stars': 2,
  'review': "She is a strict grader and doesn't provide much feedback on assignments."},
 {'professor': 'Dr. Robert Brown',
  'subject': 'Economics 101',
  'stars': 4,
  'review': 'Good professor, but you need to put in

In [12]:
processed_data = []
client = OpenAI()

for review in data['reviews']:
    response = client.embeddings.create(
        input = review['review'],
        model = "text-embedding-3-small"
    )
    embedding = response.data[0].embedding
    processed_data.append({
        "values": embedding,
        'id': review['professor'],
        'metadata': {
            'review': review['review'],
            'subject': review['subject'],
            'stars': review['stars']
        }
    })

In [14]:
processed_data[9]

{'values': [-0.008450456,
  -0.029566348,
  -0.006578649,
  -0.015275035,
  0.02415587,
  0.014892477,
  0.00842313,
  0.02251633,
  0.022215748,
  -0.0036855463,
  -0.01660033,
  -0.012378517,
  -0.023827963,
  -0.011189851,
  -0.01055453,
  -0.008013246,
  -0.015138407,
  -0.049923953,
  0.0012288,
  0.054815244,
  0.037053574,
  0.007091005,
  0.052929774,
  0.024483778,
  -0.025481163,
  -0.034621593,
  0.017119516,
  0.03945823,
  0.044267543,
  0.029757626,
  0.0678769,
  -0.01108738,
  0.005208951,
  -0.02836402,
  -0.055361755,
  0.04967802,
  0.0046316967,
  0.031506468,
  0.032107633,
  -0.0027530587,
  0.0017454255,
  0.02697041,
  -0.057875715,
  -0.019647138,
  0.04700011,
  -0.022844238,
  -0.02836402,
  -0.04899488,
  0.030905304,
  0.03694427,
  -0.05683734,
  0.066729225,
  0.07782344,
  0.0042593847,
  -0.028172739,
  -0.030386116,
  -0.018704403,
  0.03877509,
  -0.034676243,
  -0.015616606,
  0.080556,
  -0.01675062,
  -0.018116903,
  -0.030058209,
  -0.053749543,
 

In [16]:
index = pc.Index('rag')
index.upsert(
    vectors = processed_data,
    namespace = 'ns1'
)

{'upserted_count': 20}

In [17]:
index.describe_index_stats()

{'dimension': 1536,
 'index_fullness': 0.0,
 'namespaces': {'ns1': {'vector_count': 20}},
 'total_vector_count': 20}